# Entrainement modele YOLO handball

Fine-tune un YOLO26 sur un export **Handball Detection** (Roboflow Universe) :
4 classes `players` / `goalkeeper` / `referees` / `ball`.

Objectif : remplacer le YOLO26m COCO generique (qui detecte des 'person' et fragmente
le tracking) par un detecteur specialise handball qui distingue joueurs, gardiens,
arbitres et ballon.

**Sortie** : `best.pt` sauvegardé dans `models/`, a passer ensuite a
`ModeleConfig.pour_handball(...)` dans le notebook d'inference.

**Prerequis** : GPU CUDA.

## 1. Préparer l’environnement

Dans le dépôt local ou le devcontainer, utilise l’environnement Python du projet.

Pour télécharger un export absent, définis `ROBOFLOW_API_KEY` dans ton environnement. Les exports déjà présents sont utilisables sans clé.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("YOLO_CONFIG_DIR", "/tmp")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Active un GPU CUDA dans ton environnement.")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

## 2. Sélectionner le dataset

`Handball-Detection-1` est l’export utilisé pour le fine-tuning panorama, sous licence CC BY 4.0.

- [Handball-Detection-1, espace erics-workspace-it3ft](https://universe.roboflow.com/erics-workspace-it3ft/handball-detection-fj8rc-omtda/dataset/3)

La provenance et les instructions d’installation manuelle figurent dans `data/datasets/README.md`.

In [ ]:
from getpass import getpass

EXPORTS = {
    "Handball-Detection-1": ("erics-workspace-it3ft", "handball-detection-fj8rc-omtda", 3),
}
EXPORT = "Handball-Detection-1"

courant = Path.cwd().resolve()
racine = next(
    (p for p in (courant, *courant.parents)
     if (p / "scripts" / "build_train_notebook.py").is_file()),
    courant,
)
dossier_dataset = racine / "data" / "datasets" / EXPORT
if not (dossier_dataset / "data.yaml").is_file():
    try:
        from roboflow import Roboflow
    except ImportError as exc:
        raise RuntimeError("Installe roboflow pour télécharger un export absent.") from exc

    cle = os.environ.get("ROBOFLOW_API_KEY")
    if not cle:
        cle = getpass("Clé API Roboflow : ")
    workspace, projet, version = EXPORTS[EXPORT]
    rf = Roboflow(api_key=cle)
    dossier_dataset.parent.mkdir(parents=True, exist_ok=True)
    rf.workspace(workspace).project(projet).version(version).download(
        "yolov8", location=str(dossier_dataset)
    )

print(f"Dataset : {dossier_dataset}")

## 3. Inspecter le dataset

In [ ]:
from tempfile import TemporaryDirectory

import yaml

cfg = yaml.safe_load((dossier_dataset / "data.yaml").read_text(encoding="utf-8"))
cfg["path"] = str(dossier_dataset)
for cle_split, split in (("train", "train"), ("val", "valid"), ("test", "test")):
    images = dossier_dataset / split / "images"
    if not images.is_dir():
        raise FileNotFoundError(f"Dossier d’images manquant : {images}")
    cfg[cle_split] = f"{split}/images"
    print(f"{split:6s} : {sum(p.is_file() for p in images.iterdir())} images")

# Ultralytics reçoit un YAML utilisable depuis n’importe quel répertoire.
config_temporaire = TemporaryDirectory()
data_yaml = str(Path(config_temporaire.name) / "data.yaml")
Path(data_yaml).write_text(yaml.safe_dump(cfg), encoding="utf-8")
print("Classes (ordre du dataset) :", cfg["names"])

## 4. Entraîner le modèle

Le notebook reprend les poids `models/handball_<YOLO_BASE>.pt` s’ils existent et entraîne à 1280 px avec un batch automatique. Il utilise les poids COCO de `YOLO_BASE` (YOLO26m par défaut) si aucun poids local n’est disponible.

In [ ]:
from ultralytics import YOLO

YOLO_BASE = "yolo26m"
poids_locaux = racine / "models" / ("handball_"+YOLO_BASE+".pt")
MODELE_BASE = str(poids_locaux) if poids_locaux.is_file() else YOLO_BASE+".pt"
EPOCHS = 50
IMGSZ = 1280
BATCH = -1

model = YOLO(MODELE_BASE)
resultats = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=8,
    patience=15,
    name="handball_"+YOLO_BASE,
    project=str(racine / "data" / "training"),
    verbose=True,
)
print(f"Meilleurs poids : {model.trainer.best}")

## 5. Metriques de validation

In [ ]:
metrics = model.val(data=data_yaml)
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP75    : {metrics.box.map75:.4f}")
print("\nmAP50 par classe :")
for i, nom in enumerate(cfg.get("names", [])):
    try:
        print(f"  {nom:12s} : {metrics.box.ap50[i]:.4f}")
    except (IndexError, TypeError):
        pass

## 6. Exporter les poids

In [ ]:
import shutil

dossier_modeles = racine / "models"
dossier_modeles.mkdir(parents=True, exist_ok=True)
chemin_cible = dossier_modeles / ("handball_"+YOLO_BASE+".pt")
shutil.copy2(model.trainer.best, chemin_cible)
print(f"Modèle enregistré : {chemin_cible}")

## 7. Verification visuelle sur quelques images de validation

In [ ]:
import matplotlib.pyplot as plt
import cv2

images_val = [str(p) for p in sorted((dossier_dataset / "valid" / "images").iterdir()) if p.is_file()]
echantillon = images_val[:6] if len(images_val) >= 6 else images_val

if echantillon:
    preds = model.predict(echantillon, conf=0.35, verbose=False)
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    for ax, pred in zip(axes.flat, preds):
        annotee = pred.plot()  # BGR annote
        ax.imshow(cv2.cvtColor(annotee, cv2.COLOR_BGR2RGB))
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Pas d'images de validation trouvees.")

## Suite : utiliser le modele dans le pipeline

Dans le notebook d’inférence, utilise le chemin affiché à l’export pour pointer
vers le modele handball :

```python
from pivot_ai.config import ModeleConfig

config_handball = ModeleConfig.pour_handball(
    "CHEMIN_VERS_LE_MODELE_EXPORTE"
)
resultat = traiter_match_complet(
    chemin_video=CLIP,
    correspondances_homographie=correspondances,
    dossier_sortie=OUTPUTS_DIR,
    subsample=2,
    modele_config=config_handball,
)
```

Le pipeline detectera alors joueurs / gardiens / arbitres / ballon distinctement,
ce qui reduit fortement la fragmentation du tracking.